# 🔋 Battery SOH & RUL — Colab Runner

Solution 2: run from VS Code + Colab or google.colab directly.

Upload the 6 `.py` files once (cell below), then run top-to-bottom.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q kaggle scikit-learn tensorflow matplotlib seaborn pandas numpy scipy

In [ ]:
# ── Upload all 6 .py module files ─────────────────────────────────────────
# Select: config.py, imports.py, data_loader.py, feature_engineering.py,
#         model.py, inference.py, transfer_learning.py
from google.colab import files as _files
import os
uploaded = _files.upload()
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
    print(f'  ✅ {fname}')

In [ ]:
import sys
sys.path.insert(0, '/content')

from config import print_config, DATA_DIR, MODEL_PATH, SCALERS_PATH, FEATURE_COLS, SEQUENCE_LEN
print_config()

In [ ]:
# ── Kaggle credentials ─────────────────────────────────────────────────────
from google.colab import files as _files
print('Upload your kaggle.json:')
up = _files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(up['kaggle.json'])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ kaggle.json configured.')

In [ ]:
from data_loader import download_dataset, load_all_batteries
download_dataset()

In [ ]:
df_all = load_all_batteries()

In [ ]:
from feature_engineering import plot_eda
plot_eda(df_all)

In [ ]:
from feature_engineering import prepare_data
data = prepare_data(df_all)

X_train, X_val, X_test             = data['X_train'], data['X_val'], data['X_test']
y_soh_train, y_soh_val, y_soh_test = data['y_soh_train'], data['y_soh_val'], data['y_soh_test']
y_rul_train, y_rul_val, y_rul_test = data['y_rul_train'], data['y_rul_val'], data['y_rul_test']
scaler_X, scaler_soh, scaler_rul   = data['scaler_X'], data['scaler_soh'], data['scaler_rul']

In [ ]:
from model import build_cnn_lstm_model
model = build_cnn_lstm_model(SEQUENCE_LEN, len(FEATURE_COLS))
model.summary()

In [ ]:
from model import train_model
history = train_model(model, X_train, y_soh_train, y_rul_train,
                              X_val,   y_soh_val,   y_rul_val)

In [ ]:
from model import plot_training_history
plot_training_history(history)

In [ ]:
from model import evaluate_model
evaluate_model(model, X_test, y_soh_test, y_rul_test, scaler_soh, scaler_rul)

In [ ]:
from inference import save_model_and_scalers
save_model_and_scalers(model, scaler_X, scaler_soh, scaler_rul)

# Download outputs to local machine
from google.colab import files as _files
_files.download(MODEL_PATH)
_files.download(SCALERS_PATH)

## Optional: predict on your solar battery CSV

In [ ]:
# from google.colab import files as _files
# from inference import predict_solar_battery
# import pickle
#
# up = _files.upload()
# csv_name = list(up.keys())[0]
# with open(f'/content/{csv_name}', 'wb') as f:
#     f.write(up[csv_name])
#
# with open(SCALERS_PATH, 'rb') as f:
#     scalers_dict = pickle.load(f)
#
# soh_preds, rul_preds, cycles = predict_solar_battery(
#     f'/content/{csv_name}', model, scalers_dict
# )

## Optional: EV transfer learning

In [ ]:
# from google.colab import files as _files
# from transfer_learning import retrain_on_ev_battery
# import pickle
#
# up = _files.upload()
# ev_csv = list(up.keys())[0]
# with open(f'/content/{ev_csv}', 'wb') as f:
#     f.write(up[ev_csv])
#
# with open(SCALERS_PATH, 'rb') as f:
#     scalers_dict = pickle.load(f)
#
# ft_model, ev_history, ev_scalers = retrain_on_ev_battery(
#     f'/content/{ev_csv}', model, scalers_dict
# )